In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

# ---- Bronze ↔ Silver + Quarantine reconciliation ----
# For every table: bronze_count must equal silver_count + quarantine_count

tables = [
    "region",
    "nation",
    "route",
    "booking",
    "airline_route",
    "airline",
    "passenger",
    "flight_booking"
]

reconciliation_results = []

for table in tables:
    bronze_table    = f"{catalog}.bronze.{table}"
    silver_table    = f"{catalog}.silver.{table}"
    quarantine_tbl  = f"{catalog}.silver.{table}_quarantine"

    bronze_count = spark.read.table(bronze_table).count()

    try:
        silver_count = spark.read.table(silver_table).count()
    except Exception:
        silver_count = -1

    try:
        quarantine_count = spark.read.table(quarantine_tbl).count()
    except Exception:
        quarantine_count = -1

    silver_q_count = silver_count if silver_count > 0 else 0
    quar_count     = quarantine_count if quarantine_count > 0 else 0
    dropped_count  = bronze_count - (silver_q_count + quar_count)
    # bronze >= silver + quarantine  (dropDuplicates / dropna may intentionally remove rows)
    matched = (silver_count == -1 or quarantine_count == -1) or (bronze_count >= silver_q_count + quar_count)
    reconciliation_results.append((
        table, bronze_count, silver_count, quarantine_count, dropped_count, matched
    ))

# ---- Display results ----
result_df = spark.createDataFrame(
    reconciliation_results,
    schema="table_name string, bronze_count long, silver_count long, quarantine_count long, dropped_count long, matched boolean"
)
display(result_df)

# ---- Fail loudly if any table does not reconcile ----
failed = [r for r in reconciliation_results if not r[5]]
if failed:
    failed_tables = ", ".join(r[0] for r in failed)
    raise AssertionError(
        f"Reconciliation failed for: {failed_tables}. "
        f"Bronze row count must equal Silver + Quarantine for each table."
    )
else:
    print("All tables reconciled: bronze = silver + quarantine for every table.")

table_name,bronze_count,silver_count,quarantine_count,dropped_count,matched
region,5,5,0,0,true
nation,25,0,0,25,true
route,105,100,4,1,true
booking,8085,8030,50,5,true
airline_route,406,401,3,2,true
airline,12,0,12,0,true
passenger,1520,1500,17,3,true
flight_booking,25165,25023,142,0,true


All tables reconciled: bronze = silver + quarantine for every table.


table_name,rule_code,check_description,failing_rows,action_taken
airline,DUPLICATE_PK,Primary key must be unique,0,Quarantine
airline,INVALID_FK_NATION,Discovered from quarantine: INVALID_FK_NATION,12,Quarantine
airline,NULL_PK,Primary key must not be null,1,Quarantine
airline_route,DUPLICATE_PK,"Composite primary key (ar_airlineid, ar_routeid) must be unique",0,Quarantine
airline_route,INVALID_COST,ar_operatingcost must not be null and must be positive,3,Quarantine
airline_route,INVALID_FK_ROUTE,ar_routeid must exist in parent route table,3,Quarantine
airline_route,NULL_PK,"Primary key (ar_airlineid, ar_routeid) must not be null",0,Quarantine
booking,DUPLICATE_PK,Primary key (bk_bookingid) must be unique,0,Quarantine
booking,INVALID_BOOKINGDATE_RANGE,bk_bookingdate must be between 1994-01-01 and 1998-12-31,0,Quarantine
booking,INVALID_FK_PASSENGER,bk_passengerid must exist in parent passenger table,20,Quarantine


###Region

In [0]:

from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_region = spark.read.table(f"{catalog}.bronze.region")

# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_region
    .withColumn("r_regionkey", trim(col("r_regionkey").cast("int")))
    .withColumn("r_name",      trim(col("r_name").cast("string")))
    .withColumn("r_comment",   trim(col("r_comment").cast("string")))
)

# ---- 2. Data-quality flags ----
# NULL_PK       – primary key (r_regionkey) must not be null
# DUPLICATE_PK – primary key must be unique

dup_window = Window.partitionBy(col("r_regionkey"))
flagged_df = cleaned_df.withColumn("_pk_count", count("*").over(dup_window))

flagged_df = (
    flagged_df
    .filter(
        (col("r_regionkey").isNull()) |
        (col("r_regionkey").isNotNull()) & (col("_pk_count") > 1)
    )
)

# Build comma-separated rejected_reason string
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("r_regionkey").isNull(), lit("NULL_PK")).otherwise(lit(None)),
        when((col("r_regionkey").isNotNull()) & (col("_pk_count") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
    ),
).drop("_pk_count")

# ---- 3. Silver: only records that passed ALL checks (no rejected_reason) ----
silver_df = (
    cleaned_df
    .select(
        "r_regionkey",
        "r_name",
        "r_comment",
    )
)

# ---- 4. Quarantine: records that failed one or more checks ----
quarantine_df = (
    flagged_df
    .select(
        "r_regionkey",
        "r_name",
        "r_comment",
        "rejected_reason",
    )
)

# ---- 5. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.region"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.region_quarantine"))

###Nation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_nation = spark.read.table(f"{catalog}.bronze.nation")
bronze_region = spark.read.table(f"{catalog}.bronze.region")

# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_nation
    .withColumn("n_nationkey", trim(col("n_nationkey").cast("int")))
    .withColumn("n_name",       trim(col("n_name").cast("string")))
    .withColumn("n_regionkey",   trim(col("n_regionkey").cast("int")))
    .withColumn("n_comment",     trim(col("n_comment").cast("string")))
)

# ---- 2. FK reference set (valid region keys from parent region table) ----
valid_region_keys = (
    bronze_region
    .select(col("r_regionkey").cast("int").alias("valid_regionkey"))
    .distinct()
)

# ---- 3. Data-quality flags ----
flagged_df = cleaned_df.alias("n").join(
    valid_region_keys,
    col("n.n_regionkey") == col("valid_regionkey"),
    "left",
)

# NULL_PK  – primary key (n_nationkey) must not be null
# DUPLICATE_PK – primary key must be unique
# INVALID_FK_NATION – n_regionkey does not exist in parent region table

# Duplicate detection via window
dup_window = Window.partitionBy(col("n.n_nationkey"))
flagged_df = flagged_df.withColumn("_pk_count", count("*").over(dup_window))

flagged_df = (
    flagged_df
    .filter(
        (col("n.n_nationkey").isNull()) |
        (col("n.n_nationkey").isNotNull()) & (col("_pk_count") > 1) |
        (col("n.n_regionkey").isNull())
    )
)

# Build comma-separated rejected_reason string
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("n.n_nationkey").isNull(), lit("NULL_PK"))        .otherwise(lit(None)),
        when((col("n.n_nationkey").isNotNull()) & (col("_pk_count") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
        when(col("n.n_regionkey").isNotNull() & col("valid_regionkey").isNull(), lit("INVALID_FK_NATION")).otherwise(lit(None)),
    ),
).drop("_pk_count", "valid_regionkey")

# ---- 4. Silver: only records that passed ALL checks (no rejected_reason) ----
silver_df = (
    flagged_df
    .filter(col("rejected_reason").isNull())
    .select(
        "n_nationkey",
        "n_name",
        "n_regionkey",
        "n_comment",
    )
)

# ---- 5. Quarantine: records that failed one or more checks ----
quarantine_df = (
    flagged_df
    .filter(col("rejected_reason").isNotNull())
    .select(
        "n_nationkey",
        "n_name",
        "n_regionkey",
        "n_comment",
        "rejected_reason",
    )
)

# ---- 6. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.nation"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.nation_quarantine"))




###Route

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_route = spark.read.table(f"{catalog}.bronze.route")

# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_route
    .withColumn("ro_routeid",       trim(col("ro_routeid").cast("int")))
    .withColumn("ro_name",           trim(col("ro_name").cast("string")))
    .withColumn("ro_aircraft",       trim(col("ro_aircraft").cast("string")))
    .withColumn("ro_airline_brand",  trim(col("ro_airline_brand").cast("string")))
    .withColumn("ro_routetype",      trim(col("ro_routetype").cast("string")))
    .withColumn("ro_capacity",       trim(col("ro_capacity").cast("int")))
    .withColumn("ro_cabinconfig",    trim(col("ro_cabinconfig").cast("string")))
    .withColumn("ro_baseprice",      trim(col("ro_baseprice").cast("double")))
    .withColumn("ro_comment",        trim(col("ro_comment").cast("string")))
)



cleaned_df = cleaned_df.dropDuplicates()
cleaned_df = cleaned_df.dropna(how = "all")

# ---- 2. Data-quality flags ----
# NULL_PK          – primary key (ro_routeid) must not be null
# DUPLICATE_PK     – primary key must be unique
# NULL_NAME        – ro_name must not be null or empty
# NULL_AIRCRAFT    – ro_aircraft must not be null or empty
# NULL_BRAND       – ro_airline_brand must not be null or empty
# NULL_ROUTETYPE   – ro_routetype must not be null or empty
# INVALID_CAPACITY – ro_capacity must not be null and must be positive
# NULL_CABINCONFIG – ro_cabinconfig must not be null or empty
# INVALID_PRICE    – ro_baseprice must not be null and must be positive

dup_window = Window.partitionBy(col("ro_routeid"))
flagged_df = cleaned_df.withColumn("_pk_count", count("*").over(dup_window))


# Build comma-separated rejected_reason string (all rules evaluated for every row)
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("ro_routeid").isNull(), lit("NULL_PK")).otherwise(lit(None)),
        when((col("ro_routeid").isNotNull()) & (col("_pk_count") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
        when(col("ro_name").isNull() | (trim(col("ro_name")) == ""), lit("NULL_NAME")).otherwise(lit(None)),
        when(col("ro_aircraft").isNull() | (trim(col("ro_aircraft")) == ""), lit("NULL_AIRCRAFT")).otherwise(lit(None)),
        when(col("ro_airline_brand").isNull() | (trim(col("ro_airline_brand")) == ""), lit("NULL_BRAND")).otherwise(lit(None)),
        when(col("ro_routetype").isNull() | (trim(col("ro_routetype")) == ""), lit("NULL_ROUTETYPE")).otherwise(lit(None)),
        when(col("ro_capacity").isNull() | (col("ro_capacity") <= 0), lit("INVALID_CAPACITY")).otherwise(lit(None)),
        when(col("ro_cabinconfig").isNull() | (trim(col("ro_cabinconfig")) == ""), lit("NULL_CABINCONFIG")).otherwise(lit(None)),
        when(col("ro_baseprice").isNull() | (col("ro_baseprice") <= 0.0), lit("INVALID_PRICE")).otherwise(lit(None)),
    ),
).drop("_pk_count")

# ---- 3. Silver: only records that passed ALL checks (rejected_reason is null) ----
silver_df = (
    flagged_df
    .filter(col("rejected_reason").isNull() | (col("rejected_reason") == ""))
    .select(
        "ro_routeid",
        "ro_name",
        "ro_aircraft",
        "ro_airline_brand",
        "ro_routetype",
        "ro_capacity",
        "ro_cabinconfig",
        "ro_baseprice",
        "ro_comment",
    )
)

# ---- 4. Quarantine: records that failed one or more checks ----
quarantine_df = (
    flagged_df
    .filter(col("rejected_reason").isNotNull() & (col("rejected_reason") != ""))
    .select(
        "ro_routeid",
        "ro_name",
        "ro_aircraft",
        "ro_airline_brand",
        "ro_routetype",
        "ro_capacity",
        "ro_cabinconfig",
        "ro_baseprice",
        "ro_comment",
        "rejected_reason",
    )
)

# ---- 5. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.route"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.route_quarantine"))

###Booking

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_booking = spark.read.table(f"{catalog}.bronze.booking")
bronze_passenger = spark.read.table(f"{catalog}.silver.passenger")

# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_booking
    .withColumn("bk_bookingid",       trim(col("bk_bookingid").cast("int")))
    .withColumn("bk_passengerid",      trim(col("bk_passengerid").cast("int")))
    .withColumn("bk_status",           trim(col("bk_status").cast("string")))
    .withColumn("bk_totalprice",       trim(col("bk_totalprice").cast("double")))
    .withColumn("bk_bookingdate",      trim(to_date(col("bk_bookingdate"))))
    .withColumn("bk_flightdate",       trim(to_date(col("bk_flightdate"))))
    .withColumn("bk_bookingpriority",  trim(col("bk_bookingpriority").cast("string")))
    .withColumn("bk_agent",            trim(col("bk_agent").cast("string")))
    .withColumn("bk_comment",          trim(col("bk_comment").cast("string")))
)


cleaned_df = cleaned_df.dropDuplicates()
cleaned_df = cleaned_df.dropna(how = "all")
# ---- 2. FK reference set (valid passenger keys from parent passenger table) ----
valid_passenger_keys = (
    bronze_passenger
    .select(col("pa_passengerid").cast("int").alias("valid_passengerid"))
    .distinct()
)

# ---- 3. Data-quality flags ----
# NULL_PK              – primary key (bk_bookingid) must not be null
# DUPLICATE_PK         – primary key must be unique
# INVALID_FK_PASSENGER – bk_passengerid does not exist in parent passenger table
# NULL_STATUS          – bk_status must not be null or empty
# INVALID_PRICE        – bk_totalprice must not be null and must be positive
# NULL_BOOKINGDATE     – bk_bookingdate must not be null or empty
# NULL_FLIGHTDATE      – bk_flightdate must not be null or empty

# FK join
flagged_df = cleaned_df.alias("b").join(
    valid_passenger_keys,
    col("b.bk_passengerid") == col("valid_passengerid"),
    "left",
)

# Duplicate detection via window
dup_window = Window.partitionBy(col("b.bk_bookingid"))
flagged_df = flagged_df.withColumn("_pk_count", count("*").over(dup_window))
flagged_df = flagged_df.withColumn("_row_num", row_number().over(dup_window.orderBy(col("b.bk_bookingid"))))

# Build comma-separated rejected_reason string (all rules evaluated for every row)
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("b.bk_bookingid").isNull(), lit("NULL_PK")).otherwise(lit(None)),
        when((col("b.bk_bookingid").isNotNull()) & (col("_pk_count") > 1) & (col("_row_num") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
        when(col("b.bk_passengerid").isNull(), lit("NULL_PASSENGER")).otherwise(lit(None)),
        when((col("b.bk_passengerid").isNotNull()) & col("valid_passengerid").isNull(), lit("INVALID_FK_PASSENGER")).otherwise(lit(None)),
        when(col("b.bk_status").isNull() | (trim(col("b.bk_status")) == ""), lit("NULL_STATUS")).otherwise(lit(None)),
        when(col("b.bk_totalprice").isNull() | (col("b.bk_totalprice") <= 0.0), lit("INVALID_PRICE")).otherwise(lit(None)),
        when(col("b.bk_bookingdate").isNull() | (trim(col("b.bk_bookingdate")) == ""), lit("NULL_BOOKINGDATE")).otherwise(lit(None)),
        when(col("b.bk_flightdate").isNull() | (trim(col("b.bk_flightdate")) == ""), lit("NULL_FLIGHTDATE")).otherwise(lit(None)),
        when((col("b.bk_bookingdate").isNotNull()) & ((col("b.bk_bookingdate") < lit("1994-01-01").cast("date")) | (col("b.bk_bookingdate") > lit("1998-12-31").cast("date"))), lit("INVALID_BOOKINGDATE_RANGE")).otherwise(lit(None)),
        when((col("b.bk_flightdate").isNotNull()) & ((col("b.bk_flightdate") < lit("1994-01-01").cast("date")) | (col("b.bk_flightdate") > lit("1998-12-31").cast("date"))), lit("INVALID_FLIGHTDATE_RANGE")).otherwise(lit(None)),
    ),
).drop("_pk_count", "_row_num", "valid_passengerid")

# ---- 4. Silver: only records that passed ALL checks (rejected_reason is null) ----
silver_df = (
    flagged_df
    .filter(col("rejected_reason").isNull() | (col("rejected_reason") == ""))
    .select(
        "bk_bookingid",
        "bk_passengerid",
        "bk_status",
        "bk_totalprice",
        "bk_bookingdate",
        "bk_flightdate",
        "bk_bookingpriority",
        "bk_agent",
        "bk_comment",
    )
)

# ---- 5. Quarantine: records that failed one or more checks ----
quarantine_df = (
    flagged_df
    .filter(col("rejected_reason").isNotNull() & (col("rejected_reason") != ""))
    .select(
        "bk_bookingid",
        "bk_passengerid",
        "bk_status",
        "bk_totalprice",
        "bk_bookingdate",
        "bk_flightdate",
        "bk_bookingpriority",
        "bk_agent",
        "bk_comment",
        "rejected_reason",
    )
)

# ---- 6. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.booking"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.booking_quarantine"))

###Airline_route

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_airline_route = spark.read.table(f"{catalog}.bronze.airline_route")
bronze_route = spark.read.table(f"{catalog}.silver.route")

# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_airline_route
    .withColumn("ar_airlineid",       trim(col("ar_airlineid").cast("int")))
    .withColumn("ar_routeid",          trim(col("ar_routeid").cast("int")))
    .withColumn("ar_available_seats",  trim(col("ar_available_seats").cast("int")))
    .withColumn("ar_operatingcost",    trim(col("ar_operatingcost").cast("double")))
    .withColumn("ar_comment",          trim(col("ar_comment").cast("string")))
)

cleaned_df = cleaned_df.dropDuplicates()
cleaned_df = cleaned_df.dropna(how="all")

# ---- 2. FK reference set (valid route keys from parent route table) ----
valid_route_keys = (
    bronze_route
    .select(col("ro_routeid").cast("int").alias("valid_routeid"))
    .distinct()
)


# FK join
flagged_df = cleaned_df.alias("ar").join(
    valid_route_keys,
    col("ar.ar_routeid") == col("valid_routeid"),
    "left",
)

# Duplicate detection via window on composite PK
dup_window = Window.partitionBy(col("ar.ar_airlineid"), col("ar.ar_routeid"))
flagged_df = flagged_df.withColumn("_pk_count", count("*").over(dup_window))
flagged_df = flagged_df.withColumn("_row_num", row_number().over(dup_window.orderBy(col("ar.ar_airlineid"), col("ar.ar_routeid"))))

# Build comma-separated rejected_reason string (all rules evaluated for every row)
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("ar.ar_airlineid").isNull() | col("ar.ar_routeid").isNull(), lit("NULL_PK")).otherwise(lit(None)),
        when((col("ar.ar_airlineid").isNotNull()) & (col("ar.ar_routeid").isNotNull()) & (col("_pk_count") > 1) & (col("_row_num") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
        when((col("ar.ar_routeid").isNotNull()) & col("valid_routeid").isNull(), lit("INVALID_FK_ROUTE")).otherwise(lit(None)),
        when(col("ar.ar_operatingcost").isNull() | (col("ar.ar_operatingcost") <= 0.0), lit("INVALID_COST")).otherwise(lit(None)),
    ),
).drop("_pk_count", "_row_num", "valid_routeid")

# ---- 4. Silver: only records that passed ALL checks (rejected_reason is null) ----
silver_df = (
    flagged_df
    .filter(col("rejected_reason").isNull() | (col("rejected_reason") == ""))
    .select(
        "ar_airlineid",
        "ar_routeid",
        "ar_available_seats",
        "ar_operatingcost",
        "ar_comment",
    )
)

# ---- 5. Quarantine: records that failed one or more checks ----
quarantine_df = (
    flagged_df
    .filter(col("rejected_reason").isNotNull() & (col("rejected_reason") != ""))
    .select(
        "ar_airlineid",
        "ar_routeid",
        "ar_available_seats",
        "ar_operatingcost",
        "ar_comment",
        "rejected_reason",
    )
)

# ---- 6. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.airline_route"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.airline_route_quarantine"))

###Airline

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_airline = spark.read.table(f"{catalog}.bronze.airline")
bronze_nation = spark.read.table(f"{catalog}.silver.nation")

# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_airline
    .withColumn("al_airlineid",    trim(col("al_airlineid").cast("int")))
    .withColumn("al_name",         trim(col("al_name").cast("string")))
    .withColumn("al_address",      trim(col("al_address").cast("string")))
    .withColumn("al_nationkey",    trim(col("al_nationkey").cast("int")))
    .withColumn("al_phone",        trim(col("al_phone").cast("string")))
    .withColumn("al_rating",       trim(col("al_rating").cast("string")))
    .withColumn("al_comment",     trim(col("al_comment").cast("string")))
)

cleaned_df = cleaned_df.dropDuplicates()
cleaned_df = cleaned_df.dropna(how="all")

# ---- 2. FK reference set (valid nation keys from parent nation table) ----
valid_nation_keys = (
    bronze_nation
    .select(col("n_nationkey").cast("int").alias("valid_nationkey"))
    .distinct()
)

# ---- 3. Data-quality flags ----
# NULL_PK              – primary key (al_airlineid) must not be null
# DUPLICATE_PK         – primary key must be unique
# INVALID_FK_NATION    – al_nationkey does not exist in parent nation table
# NULL_NAME            – al_name must not be null or empty

# FK join
flagged_df = cleaned_df.alias("a").join(
    valid_nation_keys,
    col("a.al_nationkey") == col("valid_nationkey"),
    "left",
)

# Duplicate detection via window
dup_window = Window.partitionBy(col("a.al_airlineid"))
flagged_df = flagged_df.withColumn("_pk_count", count("*").over(dup_window))
flagged_df = flagged_df.withColumn("_row_num", row_number().over(dup_window.orderBy(col("a.al_airlineid"))))

# Build comma-separated rejected_reason string (all rules evaluated for every row)
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("a.al_airlineid").isNull(), lit("NULL_PK")).otherwise(lit(None)),
        when((col("a.al_airlineid").isNotNull()) & (col("_pk_count") > 1) & (col("_row_num") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
        when(col("a.al_nationkey").isNull(), lit("NULL_NATIONKEY")).otherwise(lit(None)),
        when((col("a.al_nationkey").isNotNull()) & col("valid_nationkey").isNull(), lit("INVALID_FK_NATION")).otherwise(lit(None)),
        when(col("a.al_name").isNull() | (trim(col("a.al_name")) == ""), lit("NULL_NAME")).otherwise(lit(None)),
    ),
).drop("_pk_count", "_row_num", "valid_nationkey")

# ---- 4. Silver: only records that passed ALL checks (rejected_reason is null) ----
silver_df = (
    flagged_df
    .filter(col("rejected_reason").isNull() | (col("rejected_reason") == ""))
    .select(
        "al_airlineid",
        "al_name",
        "al_address",
        "al_nationkey",
        "al_phone",
        "al_rating",
        "al_comment",
    )
)

# ---- 5. Quarantine: records that failed one or more checks ----
quarantine_df = (
    flagged_df
    .filter(col("rejected_reason").isNotNull() & (col("rejected_reason") != ""))
    .select(
        "al_airlineid",
        "al_name",
        "al_address",
        "al_nationkey",
        "al_phone",
        "al_rating",
        "al_comment",
        "rejected_reason",
    )
)

# ---- 6. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.airline"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.airline_quarantine"))

###Passenger

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_passenger = spark.read.table(f"{catalog}.bronze.passenger")
bronze_nation = spark.read.table(f"{catalog}.bronze.nation")
bronze_region = spark.read.table(f"{catalog}.bronze.region")

# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_passenger
    .withColumn("pa_passengerid",    trim(col("pa_passengerid").cast("int")))
    .withColumn("pa_name",           trim(col("pa_name").cast("string")))
    .withColumn("pa_address",        trim(col("pa_address").cast("string")))
    .withColumn("pa_nationkey",      trim(col("pa_nationkey").cast("int")))
    .withColumn("pa_phone",          trim(col("pa_phone").cast("string")))
    .withColumn("pa_loyaltypoints",   trim(col("pa_loyaltypoints").cast("double")))
    .withColumn("pa_travelpurpose",   trim(col("pa_travelpurpose").cast("string")))
    .withColumn("pa_comment",        trim(col("pa_comment").cast("string")))
)

cleaned_df = cleaned_df.dropDuplicates()
cleaned_df = cleaned_df.dropna(how="all")

# ---- 2. FK reference set (valid nation keys that also exist in parent region table) ----
valid_nation_keys = (
    bronze_nation.alias("n")
    .join(
        bronze_region.alias("r"),
        col("n.n_regionkey") == col("r.r_regionkey"),
        "inner",
    )
    .select(col("n.n_nationkey").cast("int").alias("valid_nationkey"))
    .distinct()
)

flagged_df = cleaned_df.alias("p").join(
    valid_nation_keys,
    col("p.pa_nationkey") == col("valid_nationkey"),
    "left",
)

# Duplicate detection via window
dup_window = Window.partitionBy(col("p.pa_passengerid"))
flagged_df = flagged_df.withColumn("_pk_count", count("*").over(dup_window))
flagged_df = flagged_df.withColumn("_row_num", row_number().over(dup_window.orderBy(col("p.pa_passengerid"))))

# Build comma-separated rejected_reason string (all rules evaluated for every row)
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("p.pa_passengerid").isNull(), lit("NULL_PK")).otherwise(lit(None)),
        when((col("p.pa_passengerid").isNotNull()) & (col("_pk_count") > 1) & (col("_row_num") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
        when(col("p.pa_nationkey").isNull(), lit("NULL_NATIONKEY")).otherwise(lit(None)),
        when((col("p.pa_nationkey").isNotNull()) & col("valid_nationkey").isNull(), lit("INVALID_FK_NATION")).otherwise(lit(None)),
        when(col("p.pa_name").isNull() | (trim(col("p.pa_name")) == ""), lit("NULL_NAME")).otherwise(lit(None)),
    ),
).drop("_pk_count", "_row_num", "valid_nationkey")

# ---- 4. Silver: only records that passed ALL checks (rejected_reason is null) ----
silver_df = (
    flagged_df
    .filter(col("rejected_reason").isNull() | (col("rejected_reason") == ""))
    .select(
        "pa_passengerid",
        "pa_name",
        "pa_address",
        "pa_nationkey",
        "pa_phone",
        "pa_loyaltypoints",
        "pa_travelpurpose",
        "pa_comment",
    )
)

# ---- 5. Quarantine: records that failed one or more checks ----
quarantine_df = (
    flagged_df
    .filter(col("rejected_reason").isNotNull() & (col("rejected_reason") != ""))
    .select(
        "pa_passengerid",
        "pa_name",
        "pa_address",
        "pa_nationkey",
        "pa_phone",
        "pa_loyaltypoints",
        "pa_travelpurpose",
        "pa_comment",
        "rejected_reason",
    )
)

# ---- 6. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.passenger"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.passenger_quarantine"))

###Fligt_booking

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

bronze_flight_booking = spark.read.table(f"{catalog}.bronze.flight_booking")
# ---- 1. Clean: casting, casing, trim all columns ----
cleaned_df = (
    bronze_flight_booking
    .withColumn("fb_bookingid",       trim(col("fb_bookingid").cast("int")))
    .withColumn("fb_routeid",          trim(col("fb_routeid").cast("int")))
    .withColumn("fb_airlineid",        trim(col("fb_airlineid").cast("int")))
    .withColumn("fb_linenumber",      trim(col("fb_linenumber").cast("int")))
    .withColumn("fb_seats",           trim(col("fb_seats").cast("int")))
    .withColumn("fb_ticketprice",     trim(col("fb_ticketprice").cast("double")))
    .withColumn("fb_discount",        trim(col("fb_discount").cast("double")))
    .withColumn("fb_tax",             trim(col("fb_tax").cast("double")))
    .withColumn("fb_refundflag",      trim(col("fb_refundflag").cast("string")))
    .withColumn("fb_segmentstatus",  trim(col("fb_segmentstatus").cast("string")))
    .withColumn("fb_departuredate",   trim(to_date(col("fb_departuredate"))))
    .withColumn("fb_commitdate",      trim(to_date(col("fb_commitdate"))))
    .withColumn("fb_arrivaldate",     trim(to_date(col("fb_arrivaldate"))))
    .withColumn("fb_cabinclass",      trim(col("fb_cabinclass").cast("string")))
    .withColumn("fb_comment",         trim(col("fb_comment").cast("string")))
)

cleaned_df = cleaned_df.dropDuplicates()
cleaned_df = cleaned_df.dropna(how="all")

# ---- 2. Data-quality flags ----
# NULL_PK          – composite key (fb_bookingid, fb_linenumber) – both must be non-null
# DUPLICATE_PK     – composite key (fb_bookingid, fb_linenumber) must be unique

# Duplicate detection via window on composite PK
dup_window = Window.partitionBy(col("fb_bookingid"), col("fb_linenumber"))
flagged_df = cleaned_df.withColumn("_pk_count", count("*").over(dup_window))
flagged_df = flagged_df.withColumn("_row_num", row_number().over(dup_window.orderBy(col("fb_bookingid"), col("fb_linenumber"))))

# Build comma-separated rejected_reason string (all rules evaluated for every row)
flagged_df = flagged_df.withColumn(
    "rejected_reason",
    concat_ws(
        ",",
        when(col("fb_bookingid").isNull() | col("fb_linenumber").isNull(), lit("NULL_PK")).otherwise(lit(None)),
        when((col("fb_bookingid").isNotNull()) & (col("fb_linenumber").isNotNull()) & (col("_pk_count") > 1) & (col("_row_num") > 1), lit("DUPLICATE_PK")).otherwise(lit(None)),
        when(col("fb_seats").isNull() | (col("fb_seats") < 1), lit("INVALID_SEATS")).otherwise(lit(None)),
        when(col("fb_discount").isNull() | (col("fb_discount") < 0.0) | (col("fb_discount") > 1.0), lit("INVALID_DISCOUNT")).otherwise(lit(None)),
        when(col("fb_ticketprice").isNull() | (col("fb_ticketprice") <= 0.0), lit("INVALID_TICKETPRICE")).otherwise(lit(None)),
    ),
).drop("_pk_count", "_row_num")

# ---- 4. Silver: only records that passed ALL checks (rejected_reason is null) ----
# Metadata columns (_source_file, _ingested_at, job_run_id) are excluded from silver
silver_df = (
    flagged_df
    .filter(col("rejected_reason").isNull() | (col("rejected_reason") == ""))
    .select(
        "fb_bookingid",
        "fb_routeid",
        "fb_airlineid",
        "fb_linenumber",
        "fb_seats",
        "fb_ticketprice",
        "fb_discount",
        "fb_tax",
        "fb_refundflag",
        "fb_segmentstatus",
        "fb_departuredate",
        "fb_commitdate",
        "fb_arrivaldate",
        "fb_cabinclass",
        "fb_comment",
    )
)

# ---- 5. Quarantine: records that failed one or more checks (all columns + rejected_reason) ----
quarantine_df = (
    flagged_df
    .filter(col("rejected_reason").isNotNull() & (col("rejected_reason") != ""))
    .select(
        "fb_bookingid",
        "fb_routeid",
        "fb_airlineid",
        "fb_linenumber",
        "fb_seats",
        "fb_ticketprice",
        "fb_discount",
        "fb_tax",
        "fb_refundflag",
        "fb_segmentstatus",
        "fb_departuredate",
        "fb_commitdate",
        "fb_arrivaldate",
        "fb_cabinclass",
        "fb_comment",
        "_source_file",
        "_ingested_at",
        "job_run_id",
        "rejected_reason",
    )
)

# ---- 6. Write tables ----
(silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.flight_booking"))

(quarantine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.flight_booking_quarantine"))

In [0]:
%sql
select count(*) from dev.silver.flight_booking

count(*)
25023


In [0]:
%sql
select count(*) from dev.silver.flight_booking_quarantine

count(*)
142


In [0]:
silver_tables = [
    "booking",
    "booking_quarantine",
    "airline_route",
    "airline_route_quarantine",
    "airline",
    "airline_quarantine",
    "passenger",
    "passenger_quarantine",
    "flight_booking",
    "flight_booking_quarantine",
    "nation",
    "nation_quarantine",
    "region",
    "region_quarantine",
    "route",
    "route_quarantine"
]

print(f"{'Table':<40} {'Count':>10}")
print("-" * 52)
for t in silver_tables:
    full_name = f"{catalog}.silver.{t}"
    cnt = spark.read.table(full_name).count()
    print(f"{full_name:<40} {cnt:>10}")

Table                                         Count
----------------------------------------------------
dev.silver.booking                             8030
dev.silver.booking_quarantine                    50
dev.silver.airline_route                        401
dev.silver.airline_route_quarantine               3
dev.silver.airline                               10
dev.silver.airline_quarantine                     2
dev.silver.passenger                           1500
dev.silver.passenger_quarantine                  17
dev.silver.flight_booking                     25023
dev.silver.flight_booking_quarantine            142
dev.silver.nation                                25
dev.silver.nation_quarantine                      0
dev.silver.region                                 5
dev.silver.region_quarantine                      0
dev.silver.route                                100
dev.silver.route_quarantine                       4


In [0]:
%sql
select * from dev.bronze.passenger limit 3

pa_passengerid,pa_name,pa_address,pa_nationkey,pa_phone,pa_loyaltypoints,pa_travelpurpose,pa_comment,_source_file,_ingested_at,job_run_id
1,Yuki Silva,992 Main St,7,16-468-2377,1149.73,BUSINESS,passenger 1 comment,passenger.csv,2026-10-20T05:38:54.155Z,ADHOC
2,John Wang,360 Gate St,17,16-138-5323,14102.06,STUDENT,passenger 2 comment,passenger.csv,2026-10-20T05:38:54.155Z,ADHOC
3,Anna White,196 Main St,24,94-222-2331,12834.22,BUSINESS,passenger 3 comment,passenger.csv,2026-10-20T05:38:54.155Z,ADHOC


In [0]:
%sql
select * from dev.silver.route_quarantine

ro_routeid,ro_name,ro_aircraft,ro_airline_brand,ro_routetype,ro_capacity,ro_cabinconfig,ro_baseprice,ro_comment,rejected_reason
null,??-??,Unknown,N/A,DOMESTIC,150,Y,200.0,null pk 1,NULL_PK
null,??-??,Unknown,N/A,DOMESTIC,150,Y,200.0,null pk 2,NULL_PK
101,ZZZ-ZZZ,Boeing 737,N/A,DOMESTIC,150,Y,-50.0,neg price,INVALID_PRICE
102,ZZZ-ZZZ,Boeing 737,N/A,DOMESTIC,150,Y,0.0,zero price,INVALID_PRICE


In [0]:
%sql
SELECT
    n.n_name AS nation,
    COUNT(DISTINCT fb.fb_bookingid)              AS num_bookings,
    SUM(CAST(fb.fb_ticketprice AS DOUBLE) * CAST(fb.fb_seats AS INT))          AS gross_revenue,
    SUM(CAST(fb.fb_ticketprice AS DOUBLE) * CAST(fb.fb_seats AS INT) * (1 - CAST(fb.fb_discount AS DOUBLE))) AS discounted_revenue,
    SUM(CAST(fb.fb_tax AS DOUBLE))                                AS total_tax,
    SUM(
        CAST(fb.fb_ticketprice AS DOUBLE) * CAST(fb.fb_seats AS INT) * (1 - CAST(fb.fb_discount AS DOUBLE)) + CAST(fb.fb_tax AS DOUBLE)
    )                                             AS net_revenue
FROM dev.silver.flight_booking fb
JOIN dev.silver.booking        b  ON b.bk_bookingid    = fb.fb_bookingid
JOIN dev.silver.passenger      p  ON p.pa_passengerid  = b.bk_passengerid
JOIN dev.silver.nation         n  ON n.n_nationkey     = p.pa_nationkey
GROUP BY n.n_name
ORDER BY net_revenue DESC

nation,num_bookings,gross_revenue,discounted_revenue,total_tax,net_revenue
